# Stage 09 / Lecture 62: Messages Architecture, Tracing Metadata, and Token Telemetry

This production reference notebook implements and verifies the structured message ecosystem in **LangChain v1+**:
1. **Text Prompts vs Message Prompts**: Contrasting stateless standalone string prompts against typed multi-message histories.
2. **SystemMessage Specificity**: Quantifying how detailed persona instructions shift LLM output specificity compared to generic one-liners.
3. **HumanMessage Tracing Metadata**: Attaching user identities (`name`) and unique message identifiers (`id`) for observability.
4. **AIMessage Seeding & Token Telemetry**: Injecting synthetic `AIMessage` history turns and auditing `usage_metadata` (`input_tokens`, `output_tokens`, `total_tokens`).
5. **ToolMessage Pairing Invariant**: Constructing paired `AIMessage(tool_calls)` and `ToolMessage(tool_call_id)` sequences for grounded synthesis.


In [1]:
from typing import List, Dict, Any
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY must be configured in .env for live execution.")

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("Model and langchain.messages classes initialized successfully.")


Model and langchain.messages classes initialized successfully.


### 1. Text Prompts vs. Rich `SystemMessage` Persona Directives

Passing a raw string (`Text Prompt`) wraps it in a standalone `HumanMessage`. Adding a rich, domain-constrained `SystemMessage` enforces output format, technology stack, and architectural best practices.


In [2]:
# 1. Standalone Text Prompt
text_prompt_resp: AIMessage = model.invoke("Define LangChain in one concise plain-text sentence without emojis.")
assert isinstance(text_prompt_resp, AIMessage)
assert len(text_prompt_resp.content.strip()) > 0
print(f"Text Prompt Output: {text_prompt_resp.content}")

# 2. Detailed SystemMessage Persona vs Generic Instruction
detailed_system_msg = SystemMessage(
    content=(
        "You are a senior Python developer with expertise in web frameworks like FastAPI. "
        "Always provide concise Python code examples, avoid emojis, and include 1 production safety tip."
    )
)
coding_query = HumanMessage(content="How do I create a simple REST API endpoint?")

structured_resp: AIMessage = model.invoke([detailed_system_msg, coding_query])
assert isinstance(structured_resp, AIMessage)
assert "fastapi" in structured_resp.content.lower() or "python" in structured_resp.content.lower()

print("\nDetailed SystemMessage Output Preview:")
print(structured_resp.content[:400] + "...")


Text Prompt Output: LangChain is an open‑source framework that enables developers to build applications powered by large language models by providing modular components for data ingestion, prompt engineering, and integration with external services.



Detailed SystemMessage Output Preview:
Below is a minimal FastAPI application that exposes a single REST endpoint `/hello`.  
It demonstrates the typical structure you’ll use in a production project.

```python
# main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Hello API")

class GreetingRequest(BaseModel):
    name: str

@app.post("/hello")
async def say_hello(req: GreetingRequest...


### 2. Tracing Metadata (`name`, `id`), Manual `AIMessage` Seeding, and `usage_metadata`

Production systems inject user metadata (`name="Alice"`, `id="message_123"`) on `HumanMessage` objects for telemetry, manually seed `AIMessage` turns for tone alignment, and inspect `usage_metadata` for token cost accounting.


In [3]:
# HumanMessage with user identity and trace ID metadata
human_greeting = HumanMessage(content="Can you help me?", name="Alice", id="message_123")
assert human_greeting.name == "Alice"
assert human_greeting.id == "message_123"

# Manually constructed AIMessage priming conversation tone
seeded_ai_reply = AIMessage(content="I would be happy to help you with that question.")

conversation_history = [
    SystemMessage(content="You are a helpful assistant. Plain text only without emojis."),
    human_greeting,
    seeded_ai_reply,
    HumanMessage(content="What is 2 + 2?", name="Alice", id="message_124")
]

telemetry_resp: AIMessage = model.invoke(conversation_history)
assert "4" in telemetry_resp.content

# Inspect usage_metadata telemetry
usage: Dict[str, Any] = telemetry_resp.usage_metadata
assert usage is not None
assert usage["input_tokens"] > 0 and usage["output_tokens"] > 0
assert usage["total_tokens"] == usage["input_tokens"] + usage["output_tokens"]

print(f"Answer: {telemetry_resp.content}")
print(f"Token Usage Telemetry: {usage}")


Answer: 2 + 2 equals 4.
Token Usage Telemetry: {'input_tokens': 120, 'output_tokens': 26, 'total_tokens': 146, 'output_token_details': {'reasoning': 9}}


### 3. The `ToolMessage` Pairing Contract (`tool_call_id`)

When feeding a tool execution result back into the context window, the `ToolMessage` must reference the exact `id` declared inside the preceding `AIMessage.tool_calls` array.


In [4]:
# Simulated tool call proposal and paired execution output
call_id = "call_san_fran_weather_01"

manual_ai_tool_request = AIMessage(
    content="",
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": call_id
    }]
)

weather_result_text = "Sunny, 72F, light coastal breeze."
paired_tool_msg = ToolMessage(
    content=weather_result_text,
    tool_call_id=call_id
)

assert paired_tool_msg.tool_call_id == manual_ai_tool_request.tool_calls[0]["id"]

tool_history = [
    SystemMessage(content="Summarize tool output concisely in plain text without emojis."),
    HumanMessage(content="What is the weather in San Francisco?"),
    manual_ai_tool_request,
    paired_tool_msg
]

grounded_answer: AIMessage = model.invoke(tool_history)
assert "72" in grounded_answer.content or "sunny" in grounded_answer.content.lower()

print(f"Synthesized Grounded Response: {grounded_answer.content}")
print("\nAll Stage 09 / Lecture 62 message architecture and telemetry assertions passed.")


Synthesized Grounded Response: Sunny, 72°F, light coastal breeze.

All Stage 09 / Lecture 62 message architecture and telemetry assertions passed.
